# mcp

> Give a rishi `Chat` real hands: run any MCP server as a subprocess and hand its tools to the model, behind rishi's approval gate.

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import json, os, queue, signal, subprocess, threading, itertools, time
from fastcore.all import Path, store_attr, L, ifnone, first
from litert_lm.interfaces import Tool
from rishi.core import Chat, hitl_policy

## Tools from a schema

litert builds a tool schema by introspecting a Python function's signature and docstring. That is the wrong shape here: an MCP server hands us a JSON Schema that was written elsewhere, and we want to pass it through untouched rather than round-trip it through a fake Python signature.

litert also accepts anything implementing its `Tool` interface, which wants exactly two methods - `get_tool_description()` returning OpenAI function-tool JSON, and `execute(args)`. That is the same shape MCP's `tools/list` already speaks, so the adapter is thin.

In [ ]:
#| export
class SchemaTool(Tool):
    "A litert `Tool` defined by a JSON schema and a callable, rather than by a Python signature."
    def __init__(self,
        name:str,            # tool name the model calls
        description:str='',  # what the tool does
        parameters:dict=None,# JSON Schema for the arguments
        fn=None              # `fn(args:dict)` implementing the call
    ):
        store_attr('name,description,fn')   # litert's `Tool` sets `__slots__`, so name the attrs explicitly
        self.parameters = ifnone(parameters, {'type': 'object', 'properties': {}})

    def get_tool_description(self):
        "OpenAI function-tool JSON, which is what litert serialises into the prompt."
        return {'type': 'function', 'function': {'name': self.name, 'description': self.description,
                                                 'parameters': self.parameters}}
    def execute(self, param):
        if self.fn is None: raise NotImplementedError(f"{self.name} has no implementation")
        return self.fn(dict(param))
    def __repr__(self): return f"SchemaTool({self.name})"

In [ ]:
t = SchemaTool('echo', 'Say it back.', {'type':'object','properties':{'text':{'type':'string'}},'required':['text']},
               fn=lambda a: a['text'].upper())
test_eq(t.get_tool_description()['function']['name'], 'echo')
test_eq(t.get_tool_description()['function']['parameters']['required'], ['text'])
test_eq(t.execute({'text':'hi'}), 'HI')

Small models are loose about JSON types: the README notes that a tool asking for an `int` receives `21.0`. A Python tool can cast and move on, but an MCP server deserialises against its schema and refuses - buzz-dev-mcp answers `invalid type: floating point 30000.0, expected u64` and the turn is wasted. `_coerce` repairs the arguments on the way out, using the schema the server itself published.

The types matter here. buzz-dev-mcp's schemas come from Rust's `schemars`, so an optional integer is `{"type": ["integer", "null"]}`, not `{"type": "integer"}` - a coercion that only understands scalar types silently does nothing.

In [ ]:
#| export
def _types(spec):
    "The JSON types a property accepts, ignoring `null` (schemars writes optionals as `['integer','null']`)."
    t = (spec or {}).get('type')
    return [x for x in (t if isinstance(t, list) else [t]) if x and x != 'null']

def _coerce(args, schema):
    "Cast model-supplied values to the JSON types `schema` asks for (models emit `21.0` for `21`)."
    props = (schema or {}).get('properties', {})
    out = {}
    for k, v in (args or {}).items():
        ts = _types(props.get(k))
        if v is None or not ts: out[k] = v; continue
        if 'integer' in ts and isinstance(v, float) and v.is_integer(): v = int(v)
        elif 'number' in ts and isinstance(v, str):
            try: v = float(v)
            except ValueError: pass
        elif 'string' in ts and not isinstance(v, str):
            v = json.dumps(v) if isinstance(v, (list, dict)) else str(v)
        out[k] = v
    return out

In [ ]:
# the real shell schema, as buzz-dev-mcp publishes it
sch = {'properties': {'timeout_ms': {'type': ['integer','null'], 'format':'uint64'},
                      'command': {'type':'string'}, 'workdir': {'type': ['string','null']}}}
test_eq(_coerce({'command':'ls', 'timeout_ms': 30000.0}, sch), {'command':'ls', 'timeout_ms': 30000})
test_eq(_coerce({'timeout_ms': None}, sch), {'timeout_ms': None})     # null stays null
test_eq(_coerce({'timeout_ms': 5500.5}, sch), {'timeout_ms': 5500.5}) # not integral: left alone
test_eq(_coerce({'command': ['ls','-l']}, sch), {'command': '["ls", "-l"]'})
test_eq(_coerce({'other': 1.0}, sch), {'other': 1.0})                 # unknown key: untouched

## An MCP server over stdio

MCP's stdio transport is JSON-RPC 2.0 with one JSON value per line: write a request to the child's stdin, read its reply from stdout. The handshake is `initialize`, then an `initialized` notification, and then the connection is live.

Two details are borrowed from `buzz-agent`, which solved them already:

- **The child's environment is a whitelist.** Only `PATH`, `HOME`, `TERM`, `LANG`, `LC_ALL` and `TMPDIR` are inherited, plus whatever you pass explicitly. Any API keys in your shell stay out of the tool subprocess.
- **The child gets its own process group** (`start_new_session=True`), so closing the client kills the whole tree rather than orphaning grandchildren that a shell tool spawned.

In [ ]:
#| export
class MCPError(Exception): "An MCP server returned an error, timed out, or died."

_ENV_KEEP = ('PATH', 'HOME', 'TERM', 'LANG', 'LC_ALL', 'TMPDIR')

def _child_env(env=None):
    "Whitelisted environment for an MCP child, so local secrets do not leak into tools."
    keep = {k: v for k in _ENV_KEEP if (v := os.environ.get(k)) is not None}
    return {**keep, **(env or {})}

In [ ]:
#| export
class MCPClient:
    "A stdio MCP server running as a subprocess, e.g. `buzz-dev-mcp`."
    def __init__(self,
        cmd:str,               # server executable
        args=(),               # its argv
        env:dict=None,         # extra environment (added to the whitelist)
        cwd=None,              # working directory - this is the workspace the tools act on
        timeout:float=180,     # seconds to wait for any one reply
        protocol:str='2025-06-18', # MCP protocol version to request
        log:bool=False,        # let the server's stderr through to yours
        name:str=None          # label, defaults to the executable name
    ):
        store_attr()
        self.name = name or Path(cmd).name
        self.proc, self.info, self.instructions = None, {}, ''
        self._ids, self._lock, self._q = itertools.count(1), threading.Lock(), queue.Queue()

    def start(self):
        "Spawn the server and complete the MCP handshake; idempotent."
        if self.proc: return self
        self.proc = subprocess.Popen([str(self.cmd), *map(str, self.args)],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            stderr=None if self.log else subprocess.DEVNULL,
            text=True, bufsize=1, env=_child_env(self.env), cwd=self.cwd, start_new_session=True)
        threading.Thread(target=self._reader, daemon=True).start()
        r = self._rpc('initialize', {'protocolVersion': self.protocol, 'capabilities': {},
                                     'clientInfo': {'name': 'rishi', 'version': '0.0.2'}})
        self.info, self.instructions = r.get('serverInfo', {}), r.get('instructions', '')
        self._notify('notifications/initialized')
        return self

    def _reader(self):
        for line in self.proc.stdout: self._q.put(line)
        self._q.put(None)

    def _send(self, o):
        if not self.proc: raise MCPError(f"{self.name} is not running")
        self.proc.stdin.write(json.dumps(o) + '\n'); self.proc.stdin.flush()

    def _notify(self, method, params=None):
        self._send({'jsonrpc': '2.0', 'method': method, **({'params': params} if params else {})})

    def _rpc(self, method, params=None, timeout=None):
        "One JSON-RPC round trip; skips notifications and any non-JSON the server prints."
        i, end = next(self._ids), time.monotonic() + ifnone(timeout, self.timeout)
        with self._lock:
            self._send({'jsonrpc': '2.0', 'id': i, 'method': method, 'params': params or {}})
            while True:
                try: line = self._q.get(timeout=max(0., end - time.monotonic()))
                except queue.Empty: raise MCPError(f"{self.name}: {method} timed out after {ifnone(timeout, self.timeout)}s")
                if line is None: raise MCPError(f"{self.name} exited during {method}")
                try: o = json.loads(line)
                except json.JSONDecodeError: continue
                if o.get('id') != i: continue
                if (e := o.get('error')): raise MCPError(f"{self.name}: {method}: {e.get('message', e)}")
                return o.get('result', {})

    def list_tools(self):
        "The tool definitions the server advertises."
        return self.start()._rpc('tools/list').get('tools', [])

    def call_tool(self, name, args=None, timeout=None):
        "Call `name` with `args`; returns the raw MCP result dict."
        return self.start()._rpc('tools/call', {'name': name, 'arguments': args or {}}, timeout=timeout)

    def close(self):
        "Close stdin, then kill the server's whole process group if it lingers. Idempotent."
        p, self.proc = self.proc, None
        if not p: return
        try: p.stdin.close()
        except Exception: pass
        try: p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
            except (ProcessLookupError, PermissionError): p.kill()
            p.wait(timeout=5)

    def __enter__(self): return self.start()
    def __exit__(self, *a): self.close()
    def __del__(self): self.close()
    def __repr__(self): return f"MCPClient({self.name}, {'running' if self.proc else 'stopped'})"

A tool result comes back as a list of content blocks. The model wants a string, so flatten it and mark failures in a way a small model reliably notices.

In [ ]:
#| export
def mcp_text(res, mx=None):
    "Flatten an MCP `tools/call` result into text for the model."
    if not isinstance(res, dict): return str(res)
    txt = '\n'.join(c.get('text', '') for c in res.get('content', []) if isinstance(c, dict) and c.get('type') == 'text')
    if not txt and (sc := res.get('structuredContent')) is not None: txt = json.dumps(sc)
    if res.get('isError'): txt = f"ERROR: {txt}"
    if mx and len(txt) > mx: txt = txt[:mx] + ' …[truncated]'
    return txt

In [ ]:
test_eq(mcp_text({'content':[{'type':'text','text':'hello'}]}), 'hello')
test_eq(mcp_text({'content':[{'type':'text','text':'nope'}],'isError':True}), 'ERROR: nope')
test_eq(mcp_text({'structuredContent':{'a':1}}), '{"a": 1}')
test_eq(mcp_text({'content':[{'type':'text','text':'abcdef'}]}, mx=3), 'abc …[truncated]')

## Handing the tools to the model

`mcp_tools` turns everything the server advertises into `SchemaTool`s that a `Chat` accepts directly. Names beginning with `_` are skipped by default: those are agent-lifecycle hooks (`_Stop`, `_PostCompact`), not tools the model should call.

A tool can fail two ways, and servers disagree about which to use - buzz-dev-mcp reports a missing file as a JSON-RPC *error*, while a failing shell command comes back as a normal result. Either way the model should see it and get a chance to fix its own mistake, so the wrapper turns both into `ERROR: …` text. `client.call_tool` still raises, because calling it yourself is not the model guessing at a path.

In [ ]:
#| export
def mcp_tools(client, include=None, exclude=None, prefix='', mx=50_000, timeout=None):
    "litert tools for an MCP server's catalogue, ready to pass to `Chat(tools=...)`."
    def _mk(t):
        name, sch = t['name'], t.get('inputSchema') or {}
        def fn(args, _n=name, _s=sch):
            try: return mcp_text(client.call_tool(_n, _coerce(args, _s), timeout=timeout), mx)
            except MCPError as e: return f"ERROR: {e}"   # let the model retry rather than killing the turn
        return SchemaTool(prefix + name, t.get('description', ''), sch, fn=fn)
    ts = [t for t in client.list_tools() if not t['name'].startswith('_')]
    if include is not None: ts = [t for t in ts if t['name'] in include]
    if exclude is not None: ts = [t for t in ts if t['name'] not in exclude]
    return L(ts).map(_mk)

### Testing without a real server

`buzz-agent` tests against real subprocesses rather than mocks, and the same approach works here: a fake MCP server in a few lines of Python exercises the framing, the handshake, error propagation, and process cleanup.

In [ ]:
#| hide
import tempfile, textwrap
_fake_src = textwrap.dedent('''
    import json, sys
    TOOLS = [{"name": "add", "description": "Add two integers.",
              "inputSchema": {"type":"object","properties":{"a":{"type":"integer"},"b":{"type":"integer"}},
                              "required":["a","b"]}},
             {"name": "boom", "description": "Fails as a result.", "inputSchema": {"type":"object","properties":{}}},
             {"name": "gone", "description": "Fails at the protocol level.", "inputSchema": {"type":"object","properties":{}}},
             {"name": "_Stop", "description": "lifecycle hook", "inputSchema": {"type":"object","properties":{}}}]
    def send(o): sys.stdout.write(json.dumps(o) + "\\n"); sys.stdout.flush()
    for line in sys.stdin:
        try: m = json.loads(line)
        except Exception: continue
        i, meth, p = m.get("id"), m.get("method"), m.get("params") or {}
        if i is None: continue
        if meth == "initialize":
            send({"jsonrpc":"2.0","id":i,"result":{"protocolVersion":p.get("protocolVersion"),
                  "serverInfo":{"name":"fake","version":"1"},"instructions":"a fake server"}})
        elif meth == "tools/list": send({"jsonrpc":"2.0","id":i,"result":{"tools":TOOLS}})
        elif meth == "tools/call":
            n, a = p.get("name"), p.get("arguments") or {}
            if n == "boom": send({"jsonrpc":"2.0","id":i,"result":{"content":[{"type":"text","text":"kaboom"}],"isError":True}})
            elif n == "add":
                ok = all(isinstance(a.get(k), int) for k in "ab")
                txt = str(a["a"] + a["b"]) if ok else f"schema violation: {a!r}"
                send({"jsonrpc":"2.0","id":i,"result":{"content":[{"type":"text","text":txt}],"isError":not ok}})
            else: send({"jsonrpc":"2.0","id":i,"error":{"code":-32601,"message":f"no tool {n}"}})
        else: send({"jsonrpc":"2.0","id":i,"result":{}})
''')
_fake = Path(tempfile.mkdtemp())/'fake_mcp.py'
_fake.write_text(_fake_src)

In [ ]:
#| hide
import sys
from fastcore.test import test_eq, test_fail
with MCPClient(sys.executable, [_fake]) as m:
    test_eq(m.info['name'], 'fake')
    test_eq(m.instructions, 'a fake server')
    by = {t.name: t for t in mcp_tools(m)}
    test_eq(list(by), ['add', 'boom', 'gone'])                      # _Stop is filtered out
    test_eq(by['add'].execute({'a': 2, 'b': 3}), '5')
    test_eq(by['add'].execute({'a': 2.0, 'b': 3.0}), '5')           # floats coerced to ints
    test_eq(by['boom'].execute({}), 'ERROR: kaboom')                # `isError` result -> text
    assert by['gone'].execute({}).startswith('ERROR:')              # JSON-RPC error -> text too
    test_fail(lambda: m.call_tool('gone'), contains='no tool gone') # ...but calling it yourself raises
    test_eq([t.name for t in mcp_tools(m, include=['add'], prefix='dev__')], ['dev__add'])
test_eq(m.proc, None)

## buzz-dev-mcp

[buzz-dev-mcp](https://github.com/block/buzz/tree/main/crates/buzz-dev-mcp) is the MCP server from Block's [buzz](https://github.com/block/buzz) workspace. It is the piece that gives an agent hands, and it is deliberately small: a shell, a file reader, a string-replace editor, an image reader, and a todo list. It also ships `rg` and `tree` on the child's `PATH`, and it is a multicall binary, so the same executable is the `buzz` relay CLI.

Point it at a directory and that directory is the workspace:

In [ ]:
#| export
BUZZ_MODES = {'read_file': 'approved', 'view_image': 'approved', 'todo': 'approved',
              'str_replace': 'check', 'shell': 'check'}

def buzz_dev_mcp(workdir=None, path='buzz-dev-mcp', shell=None, **kw):
    "An `MCPClient` for buzz-dev-mcp, scoped to `workdir` (defaults to the current directory)."
    env = {'BUZZ_SHELL': shell} if shell else None
    return MCPClient(path, cwd=str(workdir or Path.cwd()), env=env, name='buzz-dev-mcp', **kw)

`buzz-dev-mcp`'s shell has no allowlist and no confirmation step - by design, since `buzz-agent` trusts whoever launched it. rishi's `approve` gate is exactly the missing piece, so `BUZZ_MODES` is a sane default to hand to `hitl_policy`: reads run unattended, anything that writes or executes asks first.

In [ ]:
#| export
def buzz_chat(workdir=None, path='buzz-dev-mcp', modes=None, ask=None, sp=None, tools=None, **kw):
    "A `Chat` wired to buzz-dev-mcp's tools behind rishi's approval gate. Returns `(chat, client)`."
    cli = buzz_dev_mcp(workdir, path).start()
    ts = list(mcp_tools(cli)) + list(tools or [])
    pol = hitl_policy(ifnone(modes, BUZZ_MODES), **({'ask': ask} if ask else {}))
    sp = ifnone(sp, f"You are a coding agent working in a real workspace. Use the tools to do the work "
                    f"rather than describing it, and summarise what you did.\n\n{cli.instructions}")
    return Chat(tools=ts, approve=pol, sp=sp, **kw), cli

The chat and the server are separate lifetimes - close the chat to free the model, close the client to kill the tool subprocess.

In [ ]:
#| eval: false
from rishi.core import resp_text
chat, cli = buzz_chat('/tmp/workspace', modes={**BUZZ_MODES, 'shell': 'approved'}, cache_dir='.cache/litertlm')
print(resp_text(chat("How many python files are in this project, and what is in the largest one?")))
chat.close(); cli.close()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()